# Smoke test: AF2 MSA-subsampling vs GPCRmd MD (public data only)

<a href="https://colab.research.google.com/github/joyitaf/gpcr-af2-md-ensembles/blob/main/notebooks/smoke_test_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the full analysis code of the paper on **one receptor** with **public data only**, so anyone can check that the code installs, runs and produces the expected files.

| | This smoke test | Paper |
|---|---|---|
| Receptors | 1 (default OPRD1) | 5 |
| GPCRmd replicates | 1 (public, no account) | 3 (replicates 2-3 need a GPCRmd account) |
| AF2 MSA settings | 2 (default 32:320 and 1:2) | 8 |
| AF2 models per setting | 10 (2 seeds x 5 models) | 100 (20 seeds x 5 models) |

**The metrics this produces are not the paper's numbers.** The checks at the end compare what can be compared exactly: the downloaded GPCRmd file against the checksum recorded for the paper, and the extracted receptor against the paper's atom counts.

**Runtime:** select *Runtime > Change runtime type > T4 GPU* (AF2 needs a GPU). Expect roughly 30-60 min; most of it is the GPCRmd download and AF2. Each cell prints its own elapsed time.

**Data and credit.** MD trajectories come from [GPCRmd](https://www.gpcrmd.org) (cite Rodríguez-Espigares *et al.*, *Nat. Methods* 2020, doi:10.1038/s41592-020-0884-y). Structures are predicted with [ColabFold](https://github.com/sokrypton/ColabFold) (Mirdita *et al.*, *Nat. Methods* 2022). Nothing is uploaded anywhere; all files stay in this runtime under `/content`.

## 1. Settings
Change these only if you want a different receptor or more AF2 models. Every receptor works; OPRD1 is the smallest GPCRmd system and its download checksum is recorded.

In [ ]:
REPO_URL = "https://github.com/joyitaf/gpcr-af2-md-ensembles"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
RECEPTOR = "OPRD1"  # @param ["OPRD1", "CHRM2", "ADORA2A", "ADRB2", "CNR1"]
MSA_SETTINGS = "32:320 1:2"  # @param {type:"string"}
SEEDS = "1 2"  # @param {type:"string"}
# Where AF2 models come from: "colabfold" predicts them here (GPU); "skip" stops after the MD checks.
AF2_SOURCE = "colabfold"  # @param ["colabfold", "skip"]

import time
DEPTHS, SEED_LIST = MSA_SETTINGS.split(), SEEDS.split()
WORK = "/content/gpcr-af2-md-ensembles"
T0 = time.time()
print(f"{RECEPTOR}: MSA settings {DEPTHS}, seeds {SEED_LIST} -> {5 * len(SEED_LIST)} models per setting")

## 2. Install the analysis code
Clones a tagged release and installs it with pinned minimum versions from `pyproject.toml`. Restart is not needed.

In [ ]:
%%bash -s "$REPO_URL" "$REPO_REF" "$WORK"
set -euo pipefail
if [ ! -d "$3" ]; then git clone --quiet --depth 1 --branch "$2" "$1" "$3"; fi
cd "$3"
pip install --quiet -e .
gpcr-af2md --version

In [ ]:
import subprocess, sys, importlib.metadata as md_
print("Python", sys.version.split()[0])
for p in ["numpy", "scipy", "scikit-learn", "MDAnalysis", "ProDy", "matplotlib", "biopython", "PyYAML"]:
    print(f"  {p:13s} {md_.version(p)}")
print(f"[{time.time() - T0:.0f} s]")

## 3. Download public GPCRmd files and the UniProt sequence
Only the topology, the model PDB and **replicate 1** are downloaded; these need no account. Files are checked against the SHA-256 recorded in `config/receptors.yaml` where one is recorded.

In [ ]:
%cd {WORK}
!gpcr-af2md fetch --receptors {RECEPTOR} --md-replicas public
!ls -lh data/raw/gpcrmd/*/
print(f"[{time.time() - T0:.0f} s]")

## 4. Extract the receptor
Keeps receptor atoms only, checks every residue against the canonical sequence, and re-stamps frames at the physical 0.2 ns spacing.

In [ ]:
!gpcr-af2md prepare --receptors {RECEPTOR} --md-replicas public
import json
prep = json.load(open(f"data/prepared/public/{RECEPTOR}/provenance.json"))
print(json.dumps({k: prep[k] for k in ("receptor_ca", "receptor_atoms", "frame_dt_ps")}, indent=1))
print(f"[{time.time() - T0:.0f} s]")

## 5. Predict a small AF2 ensemble (GPU)
Installs ColabFold 1.6.1 (the version used for the paper) on top of Colab's JAX, builds one MSA on the ColabFold MMseqs2 server, then runs 5 models per seed at each MSA setting with dropout, as in the paper. The server's databases change over time, so this MSA can differ from the paper's deposited MSA.

In [ ]:
if AF2_SOURCE == "colabfold":
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    if gpu.returncode:
        raise SystemExit("No GPU. Choose Runtime > Change runtime type > T4 GPU, or set AF2_SOURCE='skip'.")
    print(gpu.stdout)
    !pip install --quiet --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold@v1.6.1"
    !gpcr-af2md af2-msa --receptors {RECEPTOR}
    !gpcr-af2md af2-predict --receptors {RECEPTOR} --depths {MSA_SETTINGS} --seeds {SEEDS}
print(f"[{time.time() - T0:.0f} s]")

## 6. Run the analyses
Cartesian PCA/RMSIP, RMSF, ANM, distance-PCA states and activation descriptors; backbone dihedral PCA with the JSD calibration; then cluster occupancy and nearest-MD distances. The bootstrap needs four MSA settings and is skipped here. These are the same functions the paper used (the numerical core is checksum-locked).

In [ ]:
if AF2_SOURCE != "skip":
    n = 5 * len(SEED_LIST)
    !gpcr-af2md cartesian --receptors {RECEPTOR} --md-replicas public --depths {MSA_SETTINGS} --af2-models-per-depth {n}
    !gpcr-af2md dpca --receptors {RECEPTOR} --md-replicas public
    !gpcr-af2md states --receptors {RECEPTOR} --md-replicas public
    !gpcr-af2md summarize --receptors {RECEPTOR} --md-replicas public
print(f"[{time.time() - T0:.0f} s]")

## 7. Look at the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
if AF2_SOURCE != "skip":
    display(pd.read_csv(f"results/public/cartesian/{RECEPTOR}/metrics_by_depth.csv").round(3))
    display(pd.read_csv(f"results/public/dpca/{RECEPTOR}/metrics_by_depth.csv").round(3))
    display(Image(f"results/public/dpca/{RECEPTOR}/md_vs_af2_dpca.png", width=900))

## 8. Checks
Each line prints PASS or FAIL. The first two compare against values recorded for the paper; the rest check that the pipeline produced complete, valid output.

In [ ]:
import json, math, yaml
from pathlib import Path
from gpcr_af2md.config import file_sha256

cfg = yaml.safe_load(open("config/receptors.yaml"))[RECEPTOR]["gpcrmd"]
raw = Path(f"data/raw/gpcrmd/{cfg['record']}")
# Receptor sizes of the prepared systems used in the paper (from the cluster provenance files).
PAPER_RECEPTOR = {"CHRM2": (296, 4838), "OPRD1": (303, 4854)}
checks = []
def check(name, ok): checks.append(ok); print(("PASS " if ok else "FAIL ") + name)

for entry in (cfg["topology"], cfg["model"], cfg["replicates"][0]):
    if entry["sha256"]:
        check(f"{entry['filename']} matches the checksum recorded for the paper",
              file_sha256(raw / entry["filename"]) == entry["sha256"])
    else:
        print(f"note {entry['filename']}: no checksum recorded yet")
prep = json.load(open(f"data/prepared/public/{RECEPTOR}/provenance.json"))
if RECEPTOR in PAPER_RECEPTOR:
    check("extracted receptor has the paper's C-alpha and atom counts",
          (prep["receptor_ca"], prep["receptor_atoms"]) == PAPER_RECEPTOR[RECEPTOR])
check("prepared trajectory has 0.2-ns frames", prep["frame_dt_ps"] == 200.0)
if AF2_SOURCE != "skip":
    cart = Path(f"results/public/cartesian/{RECEPTOR}")
    dp = Path(f"results/public/dpca/{RECEPTOR}")
    prov = json.load(open(cart / "PROVENANCE.json"))
    check("numerical core is the reviewed version", prov["numerical_core_sha256"]["analyze_all_core.py"].startswith("a60c7aea"))
    m = pd.read_csv(cart / "metrics_by_depth.csv")
    check("Cartesian metrics are finite", m[["RMSIP_pca", "rmsf_corr"]].map(math.isfinite).all().all())
    d = pd.read_csv(dp / "metrics_by_depth.csv")
    check("dPCA JSD values lie in [0, 1] bits", d.js_divergence_8x8_bits.between(0, 1).all())
    check("every MSA setting was analysed", sorted(d.msa_depth) == sorted(DEPTHS))
print(f"\n{sum(checks)}/{len(checks)} checks passed  [{time.time() - T0:.0f} s total]")

## 9. Keep the outputs (optional)
Zips `results/` so you can download it from the file browser on the left.

In [ ]:
!cd {WORK} && zip -qr /content/smoke_results.zip results data/prepared/public/*/provenance.json && ls -lh /content/smoke_results.zip